<a href="https://colab.research.google.com/github/victorlavrenko/answer-engineering/blob/main/notebooks/reproduce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reproduction

## Answer Engineering: Local Trajectory Control for Protocol-Constrained Reasoning in Large Language Models

This notebook keeps the top-level flow intentionally small:
1. configure dataset/model/notebook ids
2. install the package
3. create and preload `Dataset`
4. create and preload `GenerationRuntime`
5. construct notebook subruns
6. iterate subrun tasks and call normal AE generation directly
7. build a summary with `Summary(...)`
8. optionally push telemetry with `summary.push_to_hub(...)`
9. inspect one question on one subrun with low-level `task` + `runtime.generate(...)` debugging

In [ ]:
from __future__ import annotations

DATASET_ID = "lavrenko/casefactory"
SPLIT = "test"
MODEL_ID = "OpenMeditron/Meditron3-8B"
N_EVAL = 1000
MAX_NEW_TOKENS = 1024
QUESTION_ID = None  # Set this to inspect one particular dataset question id
QUESTION_SUBRUN = 7  # Usually you want 007-trajectory-editing-orl-ssnhl-acute
VERBOSITY = 0
PUSH_TELEMETRY_TO_HF = True
HF_TELEMETRY_DATASET_ID = "lavrenko/answer-engineering"
NOTEBOOK_NAME = "reproduce.ipynb"

## Answer Engineering Package Installation

In [ ]:
import importlib.util
import os
from pathlib import Path

package = "answer_engineering"
repo = f"{package.replace('_', '-')}"

try:
    from google.colab import userdata  # type: ignore[reportMissingImports]

    t = os.getenv("GITHUB_TOKEN") or userdata.get("GITHUB_TOKEN")  # type: ignore[reportUnknownMemberType]
except Exception:
    t = os.getenv("GITHUB_TOKEN")

if Path(repo).is_dir():
    !git -C {repo} pull
elif Path.cwd().name != "notebooks":
    u = f"https://{t}@github.com/" if t else "https://github.com/"
    !git clone {u}victorlavrenko/{repo}.git
if importlib.util.find_spec(package) is None:
    target = f"{repo}[hf]" if Path(repo).is_dir() else "..[dev,hf]"
    %pip install -e {target}
    raise SystemExit("Restart runtime required")

## Dataset setup

Instantiate and preload the dataset once here so rerunning the evaluation loop does not trigger a fresh dataset download.

In [ ]:
from ae_paper_reproduction import CachedHFDataset, Dataset

dataset: Dataset = CachedHFDataset(DATASET_ID, SPLIT)
dataset.materialize()

## Generation runtime setup

Instantiate and preload the model in its own cell so you can rerun it independently while editing the notebook.

In [ ]:
from answer_engineering import GenerationRuntime

runtime = GenerationRuntime(MODEL_ID)
runtime.materialize()

## Preview extracted subruns

This is optional, but useful while editing the rule cells below.

In [ ]:
from ae_paper_reproduction import NotebookSubruns

subruns = NotebookSubruns(NOTEBOOK_NAME, dataset=dataset, model=runtime)

print(f"Loaded {len(subruns)} subruns from {NOTEBOOK_NAME}:")
for subrun in subruns:
    print(f"- {subrun.name}")

## Evaluate all extracted subruns

The loop stays in the notebook for debugging, while summary building and follow-up helpers stay in the package.

In [ ]:
from ae_paper_reproduction import (
    EvaluationPrinter,
    Progress,
    RulesetEvaluationResult,
    SubrunResult,
    SubrunTask,
)
from answer_engineering import (
    GenerationPolicy,
    GenerationRequest,
    GenerationResult,
)

printer = EvaluationPrinter(verbosity=VERBOSITY)
subresults: list[SubrunResult] = []
for i, subrun in enumerate(subruns):
    print(f"{i}. Evaluating {subrun.name}:")
    tasks: tuple[SubrunTask, ...] = subrun.select_tasks(n=N_EVAL)
    task_results: list[RulesetEvaluationResult] = []
    correct = 0
    policy = GenerationPolicy(
        rules=subrun.compiled_rules,
        system_prompt=subrun.system_prompt,
        verbosity=VERBOSITY,
        max_new_tokens=MAX_NEW_TOKENS,
    )
    progress = Progress(tasks, desc=subrun.name)
    for task in progress:
        printer.task_start(task, ruleset_name=subrun.ruleset_name)
        request = GenerationRequest(question=task.question)
        answer: GenerationResult = runtime.generate(request, policy)
        task_result = RulesetEvaluationResult(task.row, answer=answer)
        if task_result.ok:
            correct += 1
        printer.task_end(
            task,
            ruleset_name=subrun.ruleset_name,
            task_result=task_result,
        )
        task_results.append(task_result)
        progress.accuracy(correct / len(task_results))
    subresults.append(SubrunResult(subrun, task_results))

In [ ]:
from ae_paper_reproduction import Summary

result = Summary(subresults)

In [ ]:
if PUSH_TELEMETRY_TO_HF:
    result.push_to_hub(HF_TELEMETRY_DATASET_ID)

In [ ]:
print(f"Wrote group report to {result.artifact_files.group_report_md}")
for subresult in subresults:
    print(
        f"{subresult.subrun_id} ({subresult.ruleset_name}) "
        f"scope={subresult.scope_label} "
        f"accuracy={subresult.report.accuracy:.3f}"
    )

## Run one question on one subrun

Set `QUESTION_ID` and choose which subrun to inspect with `QUESTION_SUBRUN` (default: the last subrun from the notebook).

In [ ]:
if QUESTION_ID:
    subrun = subruns[QUESTION_SUBRUN]
    [task] = subrun.select_tasks(question_id=QUESTION_ID)
    request = GenerationRequest(question=task.question)
    policy = GenerationPolicy(
        rules=subrun.compiled_rules,
        system_prompt=subrun.system_prompt,
        verbosity=VERBOSITY,
        max_new_tokens=MAX_NEW_TOKENS,
    )
    answer: GenerationResult = runtime.generate(request, policy)
    task_result = RulesetEvaluationResult(task.row, answer=answer)
    single_row = {
        "subrun_id": subrun.subrun_id,
        "ruleset_name": subrun.name,
        "id": task_result.id,
        "case_type": task_result.case_type,
        "question": task_result.question,
        "gold": task_result.gold,
        "ok": task_result.ok,
        "answer": task_result.answer,
    }
    print(single_row)

# Answer Engineering Rules

## Run: baseline

## Mode: baseline

## Paper Role: primary

## Variant: baseline

- orl-ssnhl-acute
- orl-conductive-acute

## System Prompt

---



# Answer Engineering Rules

## Run: reasoning

## Mode: reasoning

## Paper Role: primary

## Variant: reasoning

- orl-ssnhl-acute
- orl-conductive-acute

## System Prompt

You are an experienced physician. Interpret the key findings before drawing conclusions. Synthesize them into a clinical assessment, then state the most appropriate management decision. Always give a concrete management action that can be started now when the case allows. Do not make referral the primary recommendation. Prioritize time-sensitive conditions when supported by the findings. Use only what is explicitly stated and keep reasoning concise.

---



# Answer Engineering Rules

## Run: replace-after

## Mode: trajectory

## Paper Role: ablation

## Variant: replace-after

- orl-ssnhl-acute

## System Prompt

You are an experienced physician. Interpret the key findings before drawing conclusions. Synthesize them into a clinical assessment, then state the most appropriate management decision. Always give a concrete management action that can be started now when the case allows. Do not make referral the primary recommendation. Prioritize time-sensitive conditions when supported by the findings. Use only what is explicitly stated and keep reasoning concise.

## Replace: sensorineural hearing loss

With:

- sudden sensorineural hearing loss
- SSNHL

Prefix:

- sudden
- abrupt
- acute
- rapid onset
- within 1-72 hours
- noticed 1-72 hours

## After: SSNHL

Add:

- This condition requires urgent treatment.
- Prompt treatment is indicated.
- Treatment should be initiated without delay.

---

# Answer Engineering Rules

## Run: global-validation

## Mode: trajectory

## Paper Role: ablation

## Variant: global-validation

- orl-ssnhl-acute

## System Prompt

You are an experienced physician. Interpret the key findings before drawing conclusions. Synthesize them into a clinical assessment, then state the most appropriate management decision. Always give a concrete management action that can be started now when the case allows. Do not make referral the primary recommendation. Prioritize time-sensitive conditions when supported by the findings. Use only what is explicitly stated and keep reasoning concise.

## Avoid (all): rationalization of a diagnosis with tests

Scope: all

Prefix (any):

- conductive
- sensorineural
- stroke
- otitis
- allergic reaction
- autoimmune
- otolaryngologist
- ENT

Postfix:

- test
- testing

## Avoid (all): contralateral conductive inference Weber

Scope: all

Prefix:

- Weber | forehead
- left || right

Postfix:

- right || left
- conductive

## Avoid (all): Rinne positive then conductive

Scope: all

Prefix:

- Rinne
- positive

Postfix: conductive

## Avoid (all): explicit Rinne positive then conductive

Scope: all

Prefix: air conduction is greater than bone conduction

Postfix: conductive

## Avoid (all): incomplete laterality then diagnosis

Scope: all

Prompt (all):

- left
- right

Prefix (incomplete):

- left
- right

Postfix (any):

- conductive
- sensorineural
- stroke
- otitis
- allergic reaction
- autoimmune
- otolaryngologist
- ENT

## Avoid (all): no fork then diagnosis

Scope: all

Prefix (none):

- fork
- Weber
- Rinne

Postfix (any):

- conductive
- sensorineural
- stroke
- otitis
- allergic reaction
- autoimmune
- otolaryngologist
- ENT

## Avoid (all): hearing loss and positive Rinne then normal

Scope: all

Prefix:

- hearing loss
- Rinne
- positive

Postfix: normal

## Avoid (all): hearing loss and explicit positive Rinne then normal

Scope: all

Prefix:

- hearing loss
- Rinne
- air conduction is greater than bone conduction

Postfix: normal

---

# Answer Engineering Rules

## Run: local-validation

## Mode: trajectory

## Paper Role: ablation

## Variant: local-validation

- orl-ssnhl-acute

## System Prompt

You are an experienced physician. Interpret the key findings before drawing conclusions. Synthesize them into a clinical assessment, then state the most appropriate management decision. Always give a concrete management action that can be started now when the case allows. Do not make referral the primary recommendation. Prioritize time-sensitive conditions when supported by the findings. Use only what is explicitly stated and keep reasoning concise.

## Avoid (prefix clause): rationalization of a diagnosis with tests

Scope: all

Prefix (any):

- conductive
- sensorineural
- stroke
- otitis
- allergic reaction
- autoimmune
- otolaryngologist
- ENT

Postfix:

- test
- testing

Fallback: The test results shall be analyzed carefully.

## Avoid (last clause): contralateral conductive inference Weber

Scope: all

Prefix:

- Weber | forehead
- left || right

Postfix:

- right || left
- conductive

Fallback:

- The Weber finding should be interpreted in relation to the affected ear.
- The Weber result should be interpreted with respect to both ears.
- The Weber lateralization should be interpreted relative to the side of symptoms.

## Avoid (last clause): Rinne positive then conductive

Scope: all

Prefix:

- Rinne
- positive

Postfix: conductive

Fallback: , which shall be analyzed carefully together with other tests.

## Avoid (last clause): explicit Rinne positive then conductive

Scope: all

Prefix: air conduction is greater than bone conduction

Postfix: conductive

Fallback: , which should be interpreted together with other tests.

## Avoid (last sentence): incomplete laterality then diagnosis

Scope: all

Prompt (all):

- left
- right

Prefix (incomplete):

- left
- right

Postfix (any):

- conductive
- sensorineural
- stroke
- otitis
- allergic reaction
- autoimmune
- otolaryngologist
- ENT

Fallback:

- Weber and Rinne findings in both left and right ears should be interpreted first.
- Tuning fork tests in both left and right ears must be evaluated before diagnosis.
- The Weber and Rinne results in both left and right ears should be analyzed first.

## Avoid (last sentence): no fork then diagnosis

Scope: all

Prefix (none):

- fork
- Weber
- Rinne

Postfix (any):

- conductive
- sensorineural
- stroke
- otitis
- allergic reaction
- autoimmune
- otolaryngologist
- ENT

Fallback:

- Weber and Rinne findings should be interpreted first.
- Tuning fork tests must be evaluated before diagnosis.
- The Weber and Rinne results should be analyzed first.

## Avoid (last clause): hearing loss and positive Rinne then normal

Scope: all

Prefix:

- hearing loss
- Rinne
- positive

Postfix: normal

Fallback: .

## Avoid (last clause): hearing loss and explicit positive Rinne then normal

Scope: all

Prefix:

- hearing loss
- Rinne
- air conduction is greater than bone conduction

Postfix: normal

Fallback: .

---

# Answer Engineering Rules

## Run: trajectory-editing

## Mode: trajectory

## Paper Role: primary

## Variant: trajectory

- orl-ssnhl-acute
- orl-conductive-acute

## System Prompt

You are an experienced physician. Interpret the key findings before drawing conclusions. Synthesize them into a clinical assessment, then state the most appropriate management decision. Always give a concrete management action that can be started now when the case allows. Do not make referral the primary recommendation. Prioritize time-sensitive conditions when supported by the findings. Use only what is explicitly stated and keep reasoning concise.

## Replace: sensorineural hearing loss

With:

- sudden sensorineural hearing loss
- SSNHL

Prefix:

- sudden
- abrupt
- acute
- rapid onset
- within 1-72 hours
- noticed 1-72 hours

## After: SSNHL

Add:

- This condition requires urgent treatment.
- Prompt treatment is indicated.
- Treatment should be initiated without delay.

## Avoid (prefix clause): rationalization of a diagnosis with tests

Scope: all

Prefix (any):

- conductive
- sensorineural
- stroke
- otitis
- allergic reaction
- autoimmune
- otolaryngologist
- ENT

Postfix:

- test
- testing

Fallback: The test results shall be analyzed carefully.

## Avoid (last clause): contralateral conductive inference Weber

Scope: all

Prefix:

- Weber | forehead
- left || right

Postfix:

- right || left
- conductive

Fallback:

- The Weber finding should be interpreted in relation to the affected ear.
- The Weber result should be interpreted with respect to both ears.
- The Weber lateralization should be interpreted relative to the side of symptoms.

## Avoid (last clause): Rinne positive then conductive

Scope: all

Prefix:

- Rinne
- positive

Postfix: conductive

Fallback: , which shall be analyzed carefully together with other tests.

## Avoid (last clause): explicit Rinne positive then conductive

Scope: all

Prefix: air conduction is greater than bone conduction

Postfix: conductive

Fallback: , which should be interpreted together with other tests.

## Avoid (last sentence): incomplete laterality then diagnosis

Scope: all

Prompt (all):

- left
- right

Prefix (incomplete):

- left
- right

Postfix (any):

- conductive
- sensorineural
- stroke
- otitis
- allergic reaction
- autoimmune
- otolaryngologist
- ENT

Fallback:

- Weber and Rinne findings in both left and right ears should be interpreted first.
- Tuning fork tests in both left and right ears must be evaluated before diagnosis.
- The Weber and Rinne results in both left and right ears should be analyzed first.

## Avoid (last sentence): no fork then diagnosis

Scope: all

Prefix (none):

- fork
- Weber
- Rinne

Postfix (any):

- conductive
- sensorineural
- stroke
- otitis
- allergic reaction
- autoimmune
- otolaryngologist
- ENT

Fallback:

- Weber and Rinne findings should be interpreted first.
- Tuning fork tests must be evaluated before diagnosis.
- The Weber and Rinne results should be analyzed first.

## Avoid (last clause): hearing loss and positive Rinne then normal

Scope: all

Prefix:

- hearing loss
- Rinne
- positive

Postfix: normal

Fallback: .

## Avoid (last clause): hearing loss and explicit positive Rinne then normal

Scope: all

Prefix:

- hearing loss
- Rinne
- air conduction is greater than bone conduction

Postfix: normal

Fallback: .

---